# Compare posterior prediction and latent mixtures

Use synthetic Bernoulli and mixture data. Update a posterior, inspect point estimates, normalise component responsibilities and complete one EM-style mean update.

Fill in each starter, then run its check cell. Restart the kernel and run all cells before submitting. Keep your interpretation in the last Markdown cell.


## 1. Update the posterior and predict

Predict a Bernoulli outcome with a Beta prior.

Implement beta_predictive for positive finite Beta shapes and integer observation counts. Reject nonpositive shapes or negative counts. Return ((posterior alpha, posterior beta), probability of next success).


In [ ]:
def beta_predictive(alpha,beta,successes,failures):
    raise NotImplementedError


In [ ]:
assert beta_predictive(2,2,3,1) == ((5,3),0.625)
assert beta_predictive(1,1,0,0) == ((1,1),0.5)
assert beta_predictive(1,1,3,0) == ((4,1),0.8)
for args in ((0,1,0,0),(1,0,0,0),(1,1,-1,0),(1,1,0,-1)):
    try:
        beta_predictive(*args)
    except ValueError:
        pass
    else:
        raise AssertionError("reject nonpositive shapes and negative counts")
print("OK")


## 2. Compare an interior mode with the mean

Contrast the interior mode and mean.

Implement beta_interior_summaries(a,b) for finite Beta parameters. Reject either shape at or below one. Return (interior mode, mean).


In [ ]:
def beta_interior_summaries(a,b):
    raise NotImplementedError


In [ ]:
import math
def close(a,b): return len(a)==len(b) and all(math.isclose(x,y,rel_tol=1e-9,abs_tol=1e-12) for x,y in zip(a,b))
assert close(beta_interior_summaries(2,2),(0.5,0.5))
assert close(beta_interior_summaries(2,5),(0.2,2/7))
assert close(beta_interior_summaries(5.5,2.5),(0.75,5.5/8))
for shapes in ((1,1), (1,3), (4,0.5)):
    try:
        beta_interior_summaries(*shapes)
    except ValueError:
        pass
    else:
        raise AssertionError("both Beta shapes must exceed one")
print("OK")


## 3. Normalise log-joint mixture scores

Compute stable component responsibilities.

Implement responsibilities(log_joint). Values are finite or negative infinity. Reject empty inputs and vectors entirely negative infinity. Return normalised probabilities without underflow for equal very negative scores.


In [ ]:
def responsibilities(log_joint):
    raise NotImplementedError


In [ ]:
import math
def close(a,b): return len(a)==len(b) and all(math.isclose(x,y,rel_tol=1e-9,abs_tol=1e-12) for x,y in zip(a,b))
assert close(responsibilities([-1000,-1000]),[0.5,0.5])
assert close(responsibilities([-2,float("-inf")]),[1,0])
assert close(responsibilities([-5]),[1])
assert close(responsibilities([-1000,0]),[0,1]), "subtract the maximum: exp(1000) overflows"
assert close(responsibilities([math.log(1)-700,math.log(3)-700]),[0.25,0.75])
for values in ([],[float("-inf")]*2):
    try:
        responsibilities(values)
    except ValueError:
        pass
    else:
        raise AssertionError("reject undefined responsibilities")
print("OK")


## 4. Update a Gaussian component mean

Update one component mean.

Implement weighted_component_mean(values,weights) for aligned lists of finite observations and finite nonnegative responsibilities. Reject unequal lengths or negative weights. Return None if the effective count is zero.


In [ ]:
def weighted_component_mean(values,weights):
    raise NotImplementedError


In [ ]:
assert weighted_component_mean([2,10],[0.75,0.25]) == 4
assert weighted_component_mean([1,3],[2,2]) == 2
assert weighted_component_mean([1,3],[0,0]) is None
assert weighted_component_mean([],[]) is None
for args in (([1],[]),([1],[-1])):
    try:
        weighted_component_mean(*args)
    except ValueError:
        pass
    else:
        raise AssertionError("reject invalid summary")
print("OK")


## 5. Fit one fixed-variance mixture step

Implement mixture_step(values,means,weights). Use unit-variance one-dimensional Gaussian components,
positive prior weights summing to one and nonempty finite observations. For each point, use log(weight)
minus half squared distance as the log-joint score. The shared Gaussian constant cancels in
responsibilities. Return (new means, new weights). If a component has zero mass, retain its old mean
and assign zero weight. For tests with positive updated weights, the helper compares observed log
likelihood before and after the exact step.


In [ ]:
def mixture_step(values,means,weights):
    raise NotImplementedError


In [ ]:
from math import exp,log
import math
def close(a,b): return len(a)==len(b) and all(math.isclose(x,y,rel_tol=1e-9,abs_tol=1e-12) for x,y in zip(a,b))
assert mixture_step([1,3],[2],[1]) == ([2.0],[1.0])
values,means,weights = [-3,-2,-1,4,5,6],[-1,2],[0.5,0.5]
new_means,new_weights = mixture_step(values,means,weights)
def relative_log_likelihood(data,centers,masses):
    return sum(log(sum(mass * exp(-(value-center)**2 / 2)
                       for center,mass in zip(centers,masses))) for value in data)
assert abs(sum(new_weights)-1) < 1e-12
assert relative_log_likelihood(values,new_means,new_weights) >= relative_log_likelihood(values,means,weights)
assert mixture_step([0],[0,1000],[0.5,0.5]) == ([0.0,1000],[1.0,0.0])
m,w = mixture_step([5],[0,10],[0.9,0.1])
assert close(w,[0.9,0.1]) and close(m,[5,5]), "use soft responsibilities, not a hard argmax"
m,w = mixture_step([1000],[0,2000],[0.5,0.5])
assert close(w,[0.5,0.5]) and close(m,[1000,1000]), "normalise log-joint scores stably"
print("OK")


## Interpretation

Explain when a posterior mean loses important uncertainty. Describe how you would detect a collapsed mixture component and what evidence distinguishes local convergence from a useful fit.

**Your answer**
